### importing dependency

In [17]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

### Load Dataset
#### 🎯 Objective

- The Telco Customer Churn dataset represents historical records of customers from a telecom company and whether they stopped using the service.
- Predict whether a telecom customer will churn (leave) or not churn, using historical customer data.
- churn(Yes) === Customer left 
- churn(No) === Customer stayed

In [2]:
telco_dataset = pd.read_csv('./Data/Telco-Customer-Churn.csv')
telco_dataset.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
telco_dataset.shape

(7043, 21)

In [4]:
telco_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [5]:
telco_dataset.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [6]:
telco_dataset['Churn'].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [7]:
# telco_dataset.groupby('Churn').mean()

### Preprocessing

In [8]:
telco_dataset.drop("customerID", axis=1, inplace=True)
telco_dataset.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [9]:
telco_dataset["TotalCharges"] = pd.to_numeric(telco_dataset["TotalCharges"], errors='coerce')
telco_dataset["TotalCharges"].fillna(telco_dataset["TotalCharges"].median(), inplace=True)

/tmp/ipykernel_18146/2217621047.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  telco_dataset["TotalCharges"].fillna(telco_dataset["TotalCharges"].median(), inplace=True)


In [10]:
telco_dataset["Churn"] = telco_dataset["Churn"].map({"Yes":1, "No":0})

In [11]:
telco_dataset = pd.get_dummies(telco_dataset, drop_first=True)

In [12]:
X = telco_dataset.drop("Churn", axis=1)
Y = telco_dataset["Churn"]

#### Train-Test Split

In [13]:
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,train_size=0.7,random_state=2)

#### Model Training

In [14]:
random_forest = RandomForestClassifier(n_estimators=100, random_state=42)
random_forest.fit(X_train,Y_train)
# now after training paramers of the model are updated according to the input test labelled dataset

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

#### Result Visualization

In [15]:
Y_prediction = random_forest.predict(X_test)

In [18]:
# accuracy score on the training data
X_train_prediction = random_forest.predict(X_train)
training_data_accuracy = accuracy_score( Y_train, X_train_prediction)
print('Accuracy score of the training data : ', training_data_accuracy)

Accuracy score of the training data :  0.9979716024340771


In [19]:
# accuracy score on the testing dataset
X_test_prediction = random_forest.predict(X_test)
testing_data_accuracy = accuracy_score(Y_test, X_test_prediction)
print('Accuracy score of the testing data : ', testing_data_accuracy)

Accuracy score of the testing data :  0.7884524372929484


In [20]:

# Confusion matrix (counts)
cm = confusion_matrix(Y_test, X_test_prediction)
print("Confusion matrix (rows=true, cols=predicted):\n", cm)

Confusion matrix (rows=true, cols=predicted):
 [[1406  163]
 [ 284  260]]


In [21]:
# 4) Classification report (precision, recall, f1 for each class)
print("\nClassification Report:")
labels = ['No', 'Yes']

print("\nClassification Report:")
print(classification_report(Y_test, X_test_prediction,
                            target_names=labels,
                            digits=4))


Classification Report:

Classification Report:
              precision    recall  f1-score   support

          No     0.8320    0.8961    0.8628      1569
         Yes     0.6147    0.4779    0.5377       544

    accuracy                         0.7885      2113
   macro avg     0.7233    0.6870    0.7003      2113
weighted avg     0.7760    0.7885    0.7791      2113



In [23]:
def print_eval(name, y_true, y_pred, prob=None):
    print(f"\n---- {name} ----")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
    print("Classification Report:\n",
          classification_report(y_true, y_pred,
                                target_names=['No','Yes'],
                                digits=4))
    if prob is not None:
        print("ROC-AUC:", roc_auc_score(y_true, prob))


In [24]:
# ---------------- (2) n_estimators experiment ----------------
print("2) n_estimators experiment")
for n in [50, 100, 500]:
    rf = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    rf.fit(X_train, Y_train)
    y_pred = rf.predict(X_test)
    y_prob = rf.predict_proba(X_test)[:,1] if hasattr(rf, "predict_proba") else None
    print_eval(f"RandomForest n_estimators={n}", Y_test, y_pred, prob=y_prob)

2) n_estimators experiment

---- RandomForest n_estimators=50 ----
Accuracy: 0.7827733080927591
Confusion Matrix:
 [[1409  160]
 [ 299  245]]
Classification Report:
               precision    recall  f1-score   support

          No     0.8249    0.8980    0.8599      1569
         Yes     0.6049    0.4504    0.5163       544

    accuracy                         0.7828      2113
   macro avg     0.7149    0.6742    0.6881      2113
weighted avg     0.7683    0.7828    0.7715      2113

ROC-AUC: 0.8017957063322461

---- RandomForest n_estimators=100 ----
Accuracy: 0.7884524372929484
Confusion Matrix:
 [[1406  163]
 [ 284  260]]
Classification Report:
               precision    recall  f1-score   support

          No     0.8320    0.8961    0.8628      1569
         Yes     0.6147    0.4779    0.5377       544

    accuracy                         0.7885      2113
   macro avg     0.7233    0.6870    0.7003      2113
weighted avg     0.7760    0.7885    0.7791      2113

ROC-AUC: 0.8

In [25]:
print("\n================ Q2: Effect of n_estimators =================")

for n in [50, 100, 500]:
    model = RandomForestClassifier(n_estimators=n, random_state=42)
    model.fit(X_train, Y_train)

    y_pred = model.predict(X_test)

    print(f"\n---- n_estimators = {n} ----")
    print("Accuracy:", accuracy_score(Y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(Y_test, y_pred))
    print(classification_report(Y_test, y_pred, target_names=['No','Yes'], digits=4))



================ Q2: Effect of n_estimators =================

---- n_estimators = 50 ----
Accuracy: 0.7827733080927591
Confusion Matrix:
 [[1409  160]
 [ 299  245]]
              precision    recall  f1-score   support

          No     0.8249    0.8980    0.8599      1569
         Yes     0.6049    0.4504    0.5163       544

    accuracy                         0.7828      2113
   macro avg     0.7149    0.6742    0.6881      2113
weighted avg     0.7683    0.7828    0.7715      2113


---- n_estimators = 100 ----
Accuracy: 0.7884524372929484
Confusion Matrix:
 [[1406  163]
 [ 284  260]]
              precision    recall  f1-score   support

          No     0.8320    0.8961    0.8628      1569
         Yes     0.6147    0.4779    0.5377       544

    accuracy                         0.7885      2113
   macro avg     0.7233    0.6870    0.7003      2113
weighted avg     0.7760    0.7885    0.7791      2113


---- n_estimators = 500 ----
Accuracy: 0.7884524372929484
Confusion Matri

In [26]:
print("\n================ Q3: Effect of max_depth =================")

for depth in [3, 4, 5]:
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=depth,
        random_state=42
    )
    model.fit(X_train, Y_train)

    y_pred = model.predict(X_test)

    print(f"\n---- max_depth = {depth} ----")
    print("Accuracy:", accuracy_score(Y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(Y_test, y_pred))
    print(classification_report(Y_test, y_pred, target_names=['No','Yes'], digits=4))



================ Q3: Effect of max_depth =================

---- max_depth = 3 ----
Accuracy: 0.7785139611926172
Confusion Matrix:
 [[1491   78]
 [ 390  154]]
              precision    recall  f1-score   support

          No     0.7927    0.9503    0.8643      1569
         Yes     0.6638    0.2831    0.3969       544

    accuracy                         0.7785      2113
   macro avg     0.7282    0.6167    0.6306      2113
weighted avg     0.7595    0.7785    0.7440      2113


---- max_depth = 4 ----
Accuracy: 0.7898722195929957
Confusion Matrix:
 [[1471   98]
 [ 346  198]]
              precision    recall  f1-score   support

          No     0.8096    0.9375    0.8689      1569
         Yes     0.6689    0.3640    0.4714       544

    accuracy                         0.7899      2113
   macro avg     0.7392    0.6508    0.6702      2113
weighted avg     0.7734    0.7899    0.7665      2113


---- max_depth = 5 ----
Accuracy: 0.7931850449597728
Confusion Matrix:
 [[1463  106]


In [27]:
from sklearn.model_selection import StratifiedKFold

print("\n================ Q4: 5-Fold Cross Validation =================")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold = 1
for train_index, val_index in skf.split(X, Y):
    X_tr, X_val = X.iloc[train_index], X.iloc[val_index]
    Y_tr, Y_val = Y.iloc[train_index], Y.iloc[val_index]

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_tr, Y_tr)

    y_val_pred = model.predict(X_val)

    print(f"\n---- Fold {fold} ----")
    print("Accuracy:", accuracy_score(Y_val, y_val_pred))
    print("Confusion Matrix:\n", confusion_matrix(Y_val, y_val_pred))
    print(classification_report(Y_val, y_val_pred, target_names=['No','Yes'], digits=4))

    fold += 1



================ Q4: 5-Fold Cross Validation =================

---- Fold 1 ----
Accuracy: 0.7934705464868701
Confusion Matrix:
 [[929 106]
 [185 189]]
              precision    recall  f1-score   support

          No     0.8339    0.8976    0.8646      1035
         Yes     0.6407    0.5053    0.5650       374

    accuracy                         0.7935      1409
   macro avg     0.7373    0.7015    0.7148      1409
weighted avg     0.7826    0.7935    0.7851      1409


---- Fold 2 ----
Accuracy: 0.7920511000709723
Confusion Matrix:
 [[921 114]
 [179 195]]
              precision    recall  f1-score   support

          No     0.8373    0.8899    0.8628      1035
         Yes     0.6311    0.5214    0.5710       374

    accuracy                         0.7921      1409
   macro avg     0.7342    0.7056    0.7169      1409
weighted avg     0.7825    0.7921    0.7853      1409


---- Fold 3 ----
Accuracy: 0.7906316536550745
Confusion Matrix:
 [[933 102]
 [193 181]]
              p